### Bag of Stories
**Description:** Detect subtopics in stories with TextTiling. 

**Contributor:** Florentina Armaselu  

In [ ]:
# Import required packages
from pathlib import Path
import pandas as pd
import datetime as dt
import os
import nltk
from nltk.tokenize import TextTilingTokenizer
nltk.download('punkt')
nltk.download('stopwords')

Define the input, output paths.

In [ ]:
# Define the base directory relative to the current script location
base_dir = os.getcwd()
# Set the path to the input data folder
input = os.path.join(base_dir, "./../data/human/grimm/")
# Set the path to the input data folder for AI-generated stories with equal length start
input_ai_gen_el_st = os.path.join(base_dir, "./../data/ai-gen/grimm/el_start/")
# Set the path to the input data folder for AI-generated stories with text-tiling start 
input_ai_gen_tt_st = os.path.join(base_dir, "./../data/ai-gen/grimm/tt_start/")
# Set the path to the output folder
output = os.path.join(base_dir, "./../data/annotated_stories/human/grimm/text-tiling/")
# Set the path to the output data folder for AI-generated stories with equal-length start and text-tiling segments
output_ai_gen_el_st_tt_seg = os.path.join(base_dir, "./../data/annotated_stories/ai-gen/grimm/el_start/text-tiling/")
# Set the path to the output data folder for AI-generated stories with text-tiling start and segments
output_ai_gen_tt_st_tt_seg = os.path.join(base_dir, "./../data/annotated_stories/ai-gen/grimm/tt_start/text-tiling/")

Define constants.

In [ ]:
# Define string constants for file naming
FN_TEXT_TILING = 'tt'

# Define string constants for annotation elements
ANN_SEG = 'Segment' # Label for segments
ANN_SEP = '-'*40 # Separator for segments

# Define string constants for story types
STORY_HUMAN = 'human' # Label for human stories
STORY_AI_GEN = 'ai-gen'  # Label for AI-generated stories

# Define string constants for story start types
STORY_EL_START = 'el_start' # Label for AI-generated stories with equal length start
STORY_TT_START = 'tt_start' # Label for AI-generated stories with text-tiling start

Define the function for subtopic detection in stories. 

In [ ]:
# Detect subtopics in stories.
def detect_subtopics(story_type=STORY_HUMAN, story_start_type=None):
    """
    Detect subtopics in stories using TextTiling.
    Args:
        story_type (str): Type of the story ('human' or 'ai-gen').
        story_start_type (str, optional): Start type for AI-generated stories ('el_start' or 'tt_start').
    """
    inp = input  # Default input path for human stories
    out = output  # Default output path for human stories

    # Set the input and output paths based on the story type and start type.
    if story_type == STORY_AI_GEN:
        if story_start_type == STORY_EL_START:
            inp = input_ai_gen_el_st
            out = output_ai_gen_el_st_tt_seg
        elif story_start_type == STORY_TT_START:
            inp = input_ai_gen_tt_st
            out = output_ai_gen_tt_st_tt_seg

    # Print the number of files to be processed.
    print(f'Start processing, {len(list(Path(inp).glob("*.txt")))} files from the {Path(inp).name} folder to be analysed.')
    cnt_processed = 0

    # Read the story files sequentially from the input folder.
    for file_path in Path(inp).glob("*.txt"):
        with open(file_path, "r", encoding="utf-8") as input_file:
            story = input_file.read()
    
        # Print the name of the file being processed.
        print(f'Processing file: {file_path.name}')

        # Initialize the TextTilingTokenizer with default parameters.
        tokenizer = TextTilingTokenizer()
        # tokenizer = TextTilingTokenizer(w=20, k=10, smoothing_width=2, smoothing_rounds=1)

        # Tokenize the story into subtopics using TextTiling.
        subtopics = tokenizer.tokenize(story)

        # Create a timestamp for the output files
        timestamp = dt.datetime.now().strftime('%Y%m%d_%H%M%S')

        # Clean the file_path name of timestamps, to shorten the file name
        clean_stem = re.sub(r'\d{8}_\d{6}', '#', file_path.stem)

        # Save the segments to the output folder.
        output_file = os.path.join(out, f"{clean_stem}_{FN_TEXT_TILING}_{timestamp}.txt")
        with open(output_file, "w", encoding="utf-8") as f:
            for i, segment in enumerate(subtopics, 1):
                if (i==1):
                    # Extract the title from the first line
                    title = segment.split('\n')[0]  
                    f.write(f"{ANN_SEG} {i-1}:\n{title.strip()}\n{ANN_SEP}\n")
                    # Remove the title from the first segment
                    segment = segment[segment.find('\n')+1:] 
                f.write(f"{ANN_SEG} {i}:\n{segment.strip()}\n{ANN_SEP}\n")
            print(f'Saved segmented story to {Path(output_file).name}')
            cnt_processed += 1

    # Print the number of files saved in the output folder.
    print(f'End processing, {cnt_processed}, files saved to {Path(out).name} folder.')

Read the stories, analyse them, and save the results.  

In [ ]:
# Choose story type for subtopic detection
story_type = STORY_HUMAN  # Options: STORY_HUMAN, STORY_AI_GEN
# Choose story start type for AI-generated stories
story_start_type = None  # Options: STORY_EL_START, STORY_TT_START, None for human stories

# Validate the story type and start type
if story_type not in [STORY_HUMAN, STORY_AI_GEN]:
    raise ValueError(f"Invalid story type: {story_type}. Choose from {STORY_HUMAN}, {STORY_AI_GEN}.")
if story_start_type not in [STORY_EL_START, STORY_TT_START, None]:
    raise ValueError(f"Invalid story start type: {story_start_type}. Choose from {STORY_EL_START}, {STORY_TT_START}, None.")

# Call the function to detect subtopics inside stories.
detect_subtopics(story_type, story_start_type)